## Import Libraries

In [ ]:
import __future__
import keras

from __future__ import print_function
from keras.utils import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding
from keras.layers import SimpleRNN
from keras.datasets import imdb
from keras import initializers

## Utility

In [ ]:
def decode_review(encoded_review):
    """การทำงานของโค๊ดนี้คือ การนำ key (ที่ผ่านการสลับ key - value มาแล้ว)ที่เป็น unique id ของคำใน dict มา map กลับเป็นคำ"""
    # Bad index ให้ใช้คำว่า "?" แทน
    decoded_review = " ".join([inv_word_index.get(i, "?") for i in encoded_review])
    return decoded_review

## Load imdb Data

In [4]:
max_features = 20000
maxlen = 80
batch_size = 32

In [5]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

17464789/17464789 [==============================] - 8s 0us/step


In [8]:
print(len(X_train), 'train sequences')
print(len(X_test), 'test sequences')

25000 train sequences
25000 test sequences


## Decode data into Sentence

In [20]:
# get_word_index เป็นการสร้าง dictionary ในการ map คำ กับ unique integer
word_index = imdb.get_word_index()
word_index["<PAD>"] = -3 # Padding คำให้ length เท่ากัน เพราะเรากำหนด max len เอาไว้
word_index["<START>"] = -2 # ระบุว่าเป็นคำเริ่มต้นของประโยค
word_index["<UNK>"] = -1 # คำที่ไม่รู้จัก หมายถึงเป็นคำที่ยังไม่ได้อยู่ใน Vocabulary
word_index["<UNUSED>"] = 0 # คำที่ไม่ได้ใช้
# + 3 เกิดจาก word_index ใช้ 3 ค่าแรกไปแล้วในการนิยามคำ ต่อจากนั้นถ้าเริ่มนับคำ ก็จะมีการ + 3 เข้าไปเพื่อไม่ให้ค่าทับกฎที่วางเอาไว้
# [ขยาย] word_index ที่มีการ map dictionary เอาไว้นั้นจะเริ่มที่ 1 ก่อน เราเลย + 3 เข้าไปด้วย

# Original word_index: {"the": 1, "and": 2}
# inv_word_index becomes: {4: "the", 5: "and"}

inv_word_index = {v + 3: k for k, v in word_index.items()}

In [21]:
print(decode_review([1, 14, 20]))

<START> this movie


In [23]:
print(decode_review(X_train[0]))
print(y_train[0])
print(decode_review(X_train[1]))
print(y_train[1])

<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for retail and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also congratulations to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the praising list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be p